In [1]:
#### import pandas as pd
from pybaseball import *

In [2]:
batter_names = {
    "Herrera": "Iván",
    "Walker": "Christian",
    "Mclain": "Matt",
    "Riley": "Austin",
    "Neto": "Zach",
    "Judge": "Aaron",
    "Robert": "Luis",
    # "Caissie": "Owen",
    # "Tovar": "Ezequiel",
    "Muncy": "Max",
    "Soderstrom": "Tyler",
    "Lopez": "Otto",
    "Langford": "Wyatt",
    "Hernández": "Teoscar",
    # "Isbel": "Kyle"
}
    

In [3]:
pitcher_names = {
# 04022026
# ARI - Ryne Nelson
# KC - Cole Ragans
# SF - Robbie Ray    
    # "Nelson": "Ryne",
    # "Ragans": "Cole",
    # "Ray": "Robbie"

# }

# 04032026
# ATH - Jeffrey Springs
# NYY - Will Warren
# ARI - Eduardo Rodriguez
# SEA - Bryan Woo
# MIA - Eury Pérez
# MIL - Chad Patric
# PHI - Aaron Nola
# TB - Joe Boyle
# TEX - Mackenzie Gore
# SF - Tyler Mahle
# HOU - Cristian Javier
# CIN - Brady Singer
# WSH - Miles Mikolas

"Springs":"Jeffrey",
"Warren":"Will",
"Rodríguez":"Eduardo",
"Woo":"Bryan",
"Pérez":"Eury",
"Patrick":"Chad",
"Nola":"Aaron",
"Boyle":"Joe",
"Gore":"Mackenzie",
"Mahle":"Tyler",
"Javier":"Cristian",
"Singer":"Brady",
"Mikolas":"Miles"
    
}


In [4]:
from pybaseball import statcast_pitcher, playerid_reverse_lookup
import pandas as pd

def get_pvb_by_id(batter_id, pitcher_id, start="2012-01-01", end="2026-03-31"):
    """
    Returns:
      - pvb: full pitch-by-pitch Statcast DataFrame for the matchup
      - summary: dictionary with names + full stat line
    """

    # 0. Look up player names from MLBAM IDs
    batter_info = playerid_reverse_lookup([batter_id]).iloc[0]
    pitcher_info = playerid_reverse_lookup([pitcher_id]).iloc[0]

    batter_name = f"{batter_info['name_first']} {batter_info['name_last']}"
    pitcher_name = f"{pitcher_info['name_first']} {pitcher_info['name_last']}"

    # 1. Pull pitcher’s Statcast data
    p_data = statcast_pitcher(start, end, pitcher_id)

    # 2. Filter for this batter
    pvb = p_data[p_data["batter"] == batter_id]

    # 3. Define events
    ab_events = ["single", "double", "triple", "home_run", "field_out"]
    hit_events = ["single", "double", "triple", "home_run"]

    # 4. Count stats
    PA = len(pvb.dropna(subset=["events"]))
    AB = sum(pvb["events"].isin(ab_events))
    H = sum(pvb["events"].isin(hit_events))
    _1B = sum(pvb["events"] == "single")
    _2B = sum(pvb["events"] == "double")
    _3B = sum(pvb["events"] == "triple")
    HR = sum(pvb["events"] == "home_run")
    BB = sum(pvb["events"] == "walk")
    K = sum(pvb["events"] == "strikeout")
    RBI = pvb["rbi"].sum() if "rbi" in pvb.columns else 0

    # 5. Rate stats
    AVG = H / AB if AB > 0 else 0
    OBP = (H + BB) / PA if PA > 0 else 0
    TB = _1B + (2 * _2B) + (3 * _3B) + (4 * HR)
    SLG = TB / AB if AB > 0 else 0
    OPS = OBP + SLG

    summary = {
        "Matchup": f"{batter_name} vs {pitcher_name}",
        "PA": PA, "AB": AB, "H": H,
        "1B": _1B, "2B": _2B, "3B": _3B, "HR": HR,
        "RBI": RBI, "BB": BB, "K": K,
        "AVG": round(AVG, 3),
        "OBP": round(OBP, 3),
        "SLG": round(SLG, 3),
        "OPS": round(OPS, 3),
    }

    return summary

In [5]:
batter_results = {}

for last, first in batter_names.items():
    # playerid_lookup handles unicode, accents, and Jr. suffixes
    df = playerid_lookup(last, first)

    if df.empty:
        batter_results[f"{first} {last}"] = None
    else:
        # Take the first match (usually correct)
        batter_results[f"{first} {last}"] = int(df.iloc[0]["key_mlbam"])

# Convert to a DataFrame if useful:
id_df = pd.DataFrame.from_dict(batter_results, orient="index", columns=["mlbam_id"])
print(id_df)

Gathering player lookup table. This may take a moment.
                   mlbam_id
Iván Herrera         671056
Christian Walker     572233
Matt Mclain          680574
Austin Riley         663586
Zach Neto            687263
Aaron Judge          592450
Luis Robert          673357
Owen Caissie         683357
Ezequiel Tovar       678662
Max Muncy            571970
Tyler Soderstrom     691016
Otto Lopez           672640
Wyatt Langford       694671
Teoscar Hernández    606192


In [6]:
pitcher_results = {}

for last, first in pitcher_names.items():
    # playerid_lookup handles unicode, accents, and Jr. suffixes
    df = playerid_lookup(last, first)

    if df.empty:
        pitcher_results[f"{first} {last}"] = None
    else:
        # Take the first match (usually correct)
        pitcher_results[f"{first} {last}"] = int(df.iloc[0]["key_mlbam"])

# Convert to a DataFrame if useful:
id_df = pd.DataFrame.from_dict(pitcher_results, orient="index", columns=["mlbam_id"])
print(id_df)

                   mlbam_id
Jeffrey Springs      605488
Will Warren          701542
Eduardo Rodríguez    593958
Bryan Woo            693433
Eury Pérez           691587
Chad Patrick         694477
Aaron Nola           605400
Joe Boyle            671212
Mackenzie Gore       669022
Tyler Mahle          641816
Cristian Javier      664299
Brady Singer         663903
Miles Mikolas        571945


In [7]:
# todays_bat = [get_pvb("Austin","Riley","Ryne","Nelson"),
# get_pvb("Luis","Robert","Robbie","Ray"),
# get_pvb("Matt","Wallner","Cole","Ragans")]

In [8]:
# todays_bat = [get_pvb_by_id(572233,605488), # Christian Walker vs Jeffrey Springs
# get_pvb_by_id(672640,701542), # Otto Lopez vs Will Warren
# get_pvb_by_id(663586,593958), # Austin Riley vs Eduardo Suarez
# get_pvb_by_id(687263,693433), # Zach Neto vs Bryan Woo
# get_pvb_by_id(592450,691587), # Aaron Judge vs Eury Perez
# get_pvb_by_id(683357,701542), # Owen Caissie vs Will Warren
# get_pvb_by_id(664728,694477), # Kyle Isbel vs Chad Patrick
# get_pvb_by_id(678662,605400), # Ezequiel Tovar vs Aaron Nola
# get_pvb_by_id(670242,671212), # Matt Wallner vs Joe Boyle
# get_pvb_by_id(680574,669022), # Matt Mcclain vs Mackenzie Gore
# get_pvb_by_id(673357,641816), # Luis Robert Jr vs Tyler Mahle
# get_pvb_by_id(691016,664299), # Tyler Soderstrom vs Cristian Javier
# get_pvb_by_id(694671,663903), # Wyatt Langford vs Brady Singer
# get_pvb_by_id(606192,571945), # Teoscar Hernandez vs Miles Mikoals

# ]

# todays_bat





In [9]:
import requests

def get_player_id_by_name(name):
    """Search MLB StatsAPI for player ID."""
    url = f"https://statsapi.mlb.com/api/v1/people/search?names={name}"
    data = requests.get(url).json()

    people = data.get("people", [])
    if not people:
        return None

    return people[0]["id"]


def get_player_team(player_name):
    """
    Returns (player_id, team_name, team_id)
    """
    player_id = get_player_id_by_name(player_name)
    if not player_id:
        return None, None, None

    # ⭐ hydrate=currentTeam ensures team info is included
    url = f"https://statsapi.mlb.com/api/v1/people/{player_id}?hydrate=currentTeam"
    data = requests.get(url).json()

    person = data.get("people", [{}])[0]
    team_info = person.get("currentTeam")

    if not team_info:
        return player_id, None, None

    return player_id, team_info["name"], team_info["id"]

In [10]:
print(get_player_team("Ivan Herrera"))

(671056, 'St. Louis Cardinals', 138)


In [11]:
import requests
from datetime import datetime, date, timedelta, timezone

def get_next_opponent_for_batter(batter_name):
    """
    Given a batter name (e.g., 'Shohei Ohtani'),
    return the nearest FUTURE game (not today's date).
    """

    # ------------------------
    # 1) Get MLBAM ID using MLB search API
    # ------------------------
    search_url = f"https://statsapi.mlb.com/api/v1/people/search?names={batter_name}"
    search_data = requests.get(search_url).json()
    people = search_data.get("people", [])

    if not people:
        return {"error": "Player not found"}

    player_id = people[0]["id"]

    # ------------------------
    # 2) Get player's team
    # ------------------------
    player_url = f"https://statsapi.mlb.com/api/v1/people/{player_id}?hydrate=currentTeam"
    pdata = requests.get(player_url).json()
    person = pdata.get("people", [{}])[0]
    team_info = person.get("currentTeam")

    if not team_info:
        return {
            "player_id": player_id,
            "team": None,
            "error": "Player has no current MLB team"
        }

    team_name = team_info["name"]
    team_id = team_info["id"]

    # ------------------------
    # 3) Pull schedule for next 7 days
    # ------------------------
    start = date.today()
    end = start + timedelta(days=7)

    sched_url = (
        f"https://statsapi.mlb.com/api/v1/schedule?"
        f"teamId={team_id}&sportId=1&startDate={start}&endDate={end}"
    )

    sdata = requests.get(sched_url).json()
    dates = sdata.get("dates", [])

    if not dates:
        return {
            "player_id": player_id,
            "team": team_name,
            "error": "No upcoming games found"
        }

    # ------------------------
    # 4) Find nearest FUTURE game (strictly after now)
    # ------------------------
    now = datetime.now(timezone.utc)

    next_game = None
    for day in dates:
        for game in day["games"]:
            game_time = datetime.fromisoformat(game["gameDate"].replace("Z", "+00:00"))

            if game_time > now:       # <-- IMPORTANT: skip today's game
                next_game = game
                break
        if next_game:
            break

    if not next_game:
        return {
            "player_id": player_id,
            "team": team_name,
            "error": "No future games past today"
        }

    # ------------------------
    # 5) Identify opponent team
    # ------------------------
    home = next_game["teams"]["home"]["team"]
    away = next_game["teams"]["away"]["team"]

    if home["id"] == team_id:
        opponent = away["name"]
    else:
        opponent = home["name"]

    game_date = next_game["gameDate"]

    # ------------------------
    # 6) Return final result
    # ------------------------
    return {
        "player": batter_name,
        "player_id": player_id,
        "team": team_name,
        "opponent": opponent,
        "game_date": game_date
    }

In [12]:
get_next_opponent_for_batter("Shohei Ohtani")

{'player': 'Shohei Ohtani',
 'player_id': 660271,
 'team': 'Los Angeles Dodgers',
 'opponent': 'Toronto Blue Jays',
 'game_date': '2026-04-08T19:07:00Z'}

In [13]:
import requests
from datetime import datetime, date, timedelta, timezone

def get_next_opponent_and_probable_pitcher(batter_name):
    """
    Given a batter name (e.g., 'Shohei Ohtani'),
    return the next future game (not today's game) and
    the probable starting pitcher for the opponent team.
    """

    # ------------------------
    # 1) Get MLBAM ID using MLB search API
    # ------------------------
    search_url = f"https://statsapi.mlb.com/api/v1/people/search?names={batter_name}"
    search_data = requests.get(search_url).json()
    people = search_data.get("people", [])

    if not people:
        return {"error": "Player not found"}

    player_id = people[0]["id"]

    # ------------------------
    # 2) Get player's team
    # ------------------------
    player_url = f"https://statsapi.mlb.com/api/v1/people/{player_id}?hydrate=currentTeam"
    pdata = requests.get(player_url).json()
    person = pdata.get("people", [{}])[0]
    team_info = person.get("currentTeam")

    if not team_info:
        return {
            "player_id": player_id,
            "team": None,
            "error": "Player has no current MLB team"
        }

    team_name = team_info["name"]
    team_id = team_info["id"]

    # ------------------------
    # 3) Pull schedule with HYDRATION for probable pitchers
    # ------------------------
    start = date.today()
    end = start + timedelta(days=7)

    sched_url = (
        f"https://statsapi.mlb.com/api/v1/schedule?"
        f"teamId={team_id}&sportId=1&startDate={start}&endDate={end}"
        f"&hydrate=probablePitcher"
    )

    sdata = requests.get(sched_url).json()
    dates = sdata.get("dates", [])

    if not dates:
        return {
            "player_id": player_id,
            "team": team_name,
            "error": "No upcoming games found"
        }

    # ------------------------
    # 4) Find nearest FUTURE game (skip today)
    # ------------------------
    now = datetime.now(timezone.utc)
    next_game = None

    for day in dates:
        for game in day["games"]:
            game_time = datetime.fromisoformat(game["gameDate"].replace("Z", "+00:00"))
            if game_time > now:
                next_game = game
                break
        if next_game:
            break

    if not next_game:
        return {
            "player_id": player_id,
            "team": team_name,
            "error": "No future games after today"
        }

    # ------------------------
    # 5) Identify opponent and probable pitcher
    # ------------------------
    home = next_game["teams"]["home"]["team"]
    away = next_game["teams"]["away"]["team"]

    # Opponent team object + probable pitcher object
    if home["id"] == team_id:
        opponent_team = away
        probable_pitcher_info = next_game["teams"]["away"].get("probablePitcher")
    else:
        opponent_team = home
        probable_pitcher_info = next_game["teams"]["home"].get("probablePitcher")

    # Extract probable pitcher details
    if probable_pitcher_info:
        probable_pitcher_name = probable_pitcher_info.get("fullName")
        probable_pitcher_id = probable_pitcher_info.get("id")
    else:
        probable_pitcher_name = None
        probable_pitcher_id = None

    return {
        "player": batter_name,
        "player_id": player_id,
        "team": team_name,
        "opponent": opponent_team.get("name"),
        "game_date": next_game["gameDate"],
        "probable_pitcher": probable_pitcher_name,
        "probable_pitcher_id": probable_pitcher_id
    }

In [14]:
get_next_opponent_and_probable_pitcher("Kazuma Okamoto")

{'player': 'Kazuma Okamoto',
 'player_id': 672960,
 'team': 'Toronto Blue Jays',
 'opponent': 'Los Angeles Dodgers',
 'game_date': '2026-04-08T19:07:00Z',
 'probable_pitcher': 'Shohei Ohtani',
 'probable_pitcher_id': 660271}

In [15]:
import requests
from datetime import datetime, date, timedelta, timezone
from pybaseball import playerid_lookup, statcast_pitcher, statcast
import pandas as pd

def get_next_matchup_summary(batter_name, start_date="2015-01-01", end_date=str(date.today())):
    """
    Returns:
      - Next opponent and probable pitcher
      - Quick historical matchup summary for batter vs pitcher
    """
    # ------------------------
    # 1) Get MLBAM ID using MLB search API
    # ------------------------
    search_url = f"https://statsapi.mlb.com/api/v1/people/search?names={batter_name}"
    search_data = requests.get(search_url).json()
    people = search_data.get("people", [])

    if not people:
        return {"error": "Player not found"}

    batter_id = people[0]["id"]

    # ------------------------
    # 2) Get player's team
    # ------------------------
    player_url = f"https://statsapi.mlb.com/api/v1/people/{batter_id}?hydrate=currentTeam"
    pdata = requests.get(player_url).json()
    person = pdata.get("people", [{}])[0]
    team_info = person.get("currentTeam")

    if not team_info:
        return {"error": "Player has no current MLB team"}

    team_name = team_info["name"]
    team_id = team_info["id"]

    # ------------------------
    # 3) Pull schedule with probable pitcher
    # ------------------------
    start = date.today()
    end = start + timedelta(days=7)

    sched_url = (
        f"https://statsapi.mlb.com/api/v1/schedule?"
        f"teamId={team_id}&sportId=1&startDate={start}&endDate={end}"
        f"&hydrate=probablePitcher"
    )

    sdata = requests.get(sched_url).json()
    dates = sdata.get("dates", [])

    if not dates:
        return {"error": "No upcoming games found"}

    # ------------------------
    # 4) Find nearest FUTURE game
    # ------------------------
    now = datetime.now(timezone.utc)
    next_game = None
    for day in dates:
        for game in day["games"]:
            game_time = datetime.fromisoformat(game["gameDate"].replace("Z", "+00:00"))
            if game_time > now:
                next_game = game
                break
        if next_game:
            break

    if not next_game:
        return {"error": "No future games after today"}

    # ------------------------
    # 5) Identify opponent and probable pitcher
    # ------------------------
    home = next_game["teams"]["home"]["team"]
    away = next_game["teams"]["away"]["team"]

    if home["id"] == team_id:
        opponent_team = away
        probable_pitcher_info = next_game["teams"]["away"].get("probablePitcher")
    else:
        opponent_team = home
        probable_pitcher_info = next_game["teams"]["home"].get("probablePitcher")

    if probable_pitcher_info:
        pitcher_name = probable_pitcher_info.get("fullName")
        pitcher_id = probable_pitcher_info.get("id")
    else:
        pitcher_name = None
        pitcher_id = None

    # ------------------------
    # 6) Fetch historical matchup from Statcast
    # ------------------------
    if pitcher_id:
        # Statcast pitcher data
        try:
            # Pull pitcher data only for efficiency
            pitcher_data = statcast_pitcher(start_date, end_date, pitcher_id)
            matchup = pitcher_data[pitcher_data["batter"] == batter_id]

            # Quick summary
            ab_events = ["single", "double", "triple", "home_run", "field_out"]
            hit_events = ["single", "double", "triple", "home_run"]

            PA = len(matchup.dropna(subset=["events"]))
            AB = sum(matchup["events"].isin(ab_events))
            H = sum(matchup["events"].isin(hit_events))
            _1B = sum(matchup["events"] == "single")
            _2B = sum(matchup["events"] == "double")
            _3B = sum(matchup["events"] == "triple")
            HR = sum(matchup["events"] == "home_run")
            BB = sum(matchup["events"] == "walk")
            K = sum(matchup["events"] == "strikeout")
            RBI = matchup["rbi"].sum() if "rbi" in matchup.columns else 0

            AVG = H / AB if AB > 0 else 0
            OBP = (H + BB) / PA if PA > 0 else 0
            TB = _1B + (2 * _2B) + (3 * _3B) + (4 * HR)
            SLG = TB / AB if AB > 0 else 0
            OPS = OBP + SLG

            summary = {
                "Matchup": f"{batter_name} vs {pitcher_name}",
                "PA": PA, "AB": AB, "H": H,
                "1B": _1B, "2B": _2B, "3B": _3B, "HR": HR,
                "RBI": RBI, "BB": BB, "K": K,
                "AVG": round(AVG, 3),
                "OBP": round(OBP, 3),
                "SLG": round(SLG, 3),
                "OPS": round(OPS, 3)
            }
        except Exception as e:
            summary = {"error": f"Failed to get matchup data: {e}"}
    else:
        summary = {"info": "No probable pitcher yet for this game"}

    # ------------------------
    # 7) Return results
    # ------------------------
    print(summary)
    return {
        "player": batter_name,
        "team": team_name,
        "opponent": opponent_team.get("name"),
        "game_date": next_game["gameDate"],
        "probable_pitcher": pitcher_name,
        "matchup_summary": summary
    }

In [16]:
get_next_matchup_summary("Iván Herrera")

Gathering Player Data
{'Matchup': 'Iván Herrera vs Miles Mikolas', 'PA': 0, 'AB': 0, 'H': 0, '1B': 0, '2B': 0, '3B': 0, 'HR': 0, 'RBI': 0, 'BB': 0, 'K': 0, 'AVG': 0, 'OBP': 0, 'SLG': 0, 'OPS': 0}


{'player': 'Iván Herrera',
 'team': 'St. Louis Cardinals',
 'opponent': 'Washington Nationals',
 'game_date': '2026-04-08T20:05:00Z',
 'probable_pitcher': 'Miles Mikolas',
 'matchup_summary': {'Matchup': 'Iván Herrera vs Miles Mikolas',
  'PA': 0,
  'AB': 0,
  'H': 0,
  '1B': 0,
  '2B': 0,
  '3B': 0,
  'HR': 0,
  'RBI': 0,
  'BB': 0,
  'K': 0,
  'AVG': 0,
  'OBP': 0,
  'SLG': 0,
  'OPS': 0}}

In [17]:
get_next_matchup_summary("Christian Herrera")

{'error': 'Player not found'}

In [18]:
batter_names = {
    "Herrera": "Iván",
    "Walker": "Christian",
    "Mclain": "Matt",
    "Riley": "Austin",
    "Neto": "Zach",
    "Judge": "Aaron",
    "Robert": "Luis",
    "Caissie": "Owen",
    "Tovar": "Ezequiel",
    "Wallner": "Matt",
    "Soderstrom": "Tyler",
    "Lopez": "Otto",
    "Langford": "Wyatt",
    "Hernández": "Teoscar",
    # "Isbel": "Kyle"
}

results = {}

for last, first in batter_names.items():
    full_name = f"{first} {last}"
    print(f"Processing {full_name} ...")
    
    try:
        result = get_next_matchup_summary(full_name)
        results[full_name] = result["matchup_summary"]
    except Exception as e:
        results[full_name] = {"error": str(e)}

# Example: print all results
for player, summary in results.items():
    print(f"\n{player}:")
    print(summary)

Processing Iván Herrera ...
Gathering Player Data
{'Matchup': 'Iván Herrera vs Miles Mikolas', 'PA': 0, 'AB': 0, 'H': 0, '1B': 0, '2B': 0, '3B': 0, 'HR': 0, 'RBI': 0, 'BB': 0, 'K': 0, 'AVG': 0, 'OBP': 0, 'SLG': 0, 'OPS': 0}
Processing Christian Walker ...
Gathering Player Data
{'Matchup': 'Christian Walker vs Michael Lorenzen', 'PA': 10, 'AB': 7, 'H': 2, '1B': 1, '2B': 1, '3B': 0, 'HR': 0, 'RBI': 0, 'BB': 2, 'K': 1, 'AVG': 0.286, 'OBP': 0.4, 'SLG': 0.429, 'OPS': 0.829}
Processing Matt Mclain ...
Gathering Player Data


C:\Users\shyun\AppData\Local\Programs\Python\Python313\Lib\site-packages\pybaseball\utils.py:299: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(results)


{'Matchup': 'Matt Mclain vs Eury Pérez', 'PA': 2, 'AB': 1, 'H': 1, '1B': 0, '2B': 0, '3B': 0, 'HR': 1, 'RBI': 0, 'BB': 0, 'K': 1, 'AVG': 1.0, 'OBP': 0.5, 'SLG': 4.0, 'OPS': 4.5}
Processing Austin Riley ...
Gathering Player Data


C:\Users\shyun\AppData\Local\Programs\Python\Python313\Lib\site-packages\pybaseball\utils.py:299: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(results)


{'Matchup': 'Austin Riley vs Reid Detmers', 'PA': 3, 'AB': 2, 'H': 2, '1B': 2, '2B': 0, '3B': 0, 'HR': 0, 'RBI': 0, 'BB': 0, 'K': 1, 'AVG': 1.0, 'OBP': 0.667, 'SLG': 1.0, 'OPS': 1.667}
Processing Zach Neto ...
Gathering Player Data
{'Matchup': 'Zach Neto vs Grant Holmes', 'PA': 3, 'AB': 1, 'H': 0, '1B': 0, '2B': 0, '3B': 0, 'HR': 0, 'RBI': 0, 'BB': 0, 'K': 2, 'AVG': 0.0, 'OBP': 0.0, 'SLG': 0.0, 'OPS': 0.0}
Processing Aaron Judge ...
Gathering Player Data
{'Matchup': 'Aaron Judge vs Luis Severino', 'PA': 6, 'AB': 3, 'H': 3, '1B': 1, '2B': 1, '3B': 0, 'HR': 1, 'RBI': 0, 'BB': 1, 'K': 2, 'AVG': 1.0, 'OBP': 0.667, 'SLG': 2.333, 'OPS': 3.0}
Processing Luis Robert ...
Gathering Player Data


...

{'Matchup': 'Luis Robert vs Ryne Nelson', 'PA': 7, 'AB': 3, 'H': 1, '1B': 0, '2B': 0, '3B': 0, 'HR': 1, 'RBI': 0, 'BB': 0, 'K': 4, 'AVG': 0.333, 'OBP': 0.143, 'SLG': 1.333, 'OPS': 1.476}
Processing Owen Caissie ...
Gathering Player Data
{'Matchup': 'Owen Caissie vs Brady Singer', 'PA': 0, 'AB': 0, 'H': 0, '1B': 0, '2B': 0, '3B': 0, 'HR': 0, 'RBI': 0, 'BB': 0, 'K': 0, 'AVG': 0, 'OBP': 0, 'SLG': 0, 'OPS': 0}
Processing Ezequiel Tovar ...
Gathering Player Data
{'Matchup': 'Ezequiel Tovar vs Cristian Javier', 'PA': 0, 'AB': 0, 'H': 0, '1B': 0, '2B': 0, '3B': 0, 'HR': 0, 'RBI': 0, 'BB': 0, 'K': 0, 'AVG': 0, 'OBP': 0, 'SLG': 0, 'OPS': 0}
Processing Matt Wallner ...
Gathering Player Data
{'Matchup': 'Matt Wallner vs Framber Valdez', 'PA': 0, 'AB': 0, 'H': 0, '1B': 0, '2B': 0, '3B': 0, 'HR': 0, 'RBI': 0, 'BB': 0, 'K': 0, 'AVG': 0, 'OBP': 0, 'SLG': 0, 'OPS': 0}
Processing Tyler Soderstrom ...
Gathering Player Data


C:\Users\shyun\AppData\Local\Programs\Python\Python313\Lib\site-packages\pybaseball\utils.py:299: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(results)


{'Matchup': 'Tyler Soderstrom vs Will Warren', 'PA': 5, 'AB': 2, 'H': 0, '1B': 0, '2B': 0, '3B': 0, 'HR': 0, 'RBI': 0, 'BB': 2, 'K': 1, 'AVG': 0.0, 'OBP': 0.4, 'SLG': 0.0, 'OPS': 0.4}
Processing Otto Lopez ...
Gathering Player Data
{'Matchup': 'Otto Lopez vs Brady Singer', 'PA': 5, 'AB': 3, 'H': 0, '1B': 0, '2B': 0, '3B': 0, 'HR': 0, 'RBI': 0, 'BB': 0, 'K': 2, 'AVG': 0.0, 'OBP': 0.0, 'SLG': 0.0, 'OPS': 0.0}
Processing Wyatt Langford ...
Gathering Player Data


C:\Users\shyun\AppData\Local\Programs\Python\Python313\Lib\site-packages\pybaseball\utils.py:299: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(results)


{'Matchup': 'Wyatt Langford vs Bryan Woo', 'PA': 6, 'AB': 3, 'H': 1, '1B': 0, '2B': 0, '3B': 0, 'HR': 1, 'RBI': 0, 'BB': 0, 'K': 3, 'AVG': 0.333, 'OBP': 0.167, 'SLG': 1.333, 'OPS': 1.5}
Processing Teoscar Hernández ...
Gathering Player Data
{'Matchup': 'Teoscar Hernández vs Dylan Cease', 'PA': 24, 'AB': 10, 'H': 4, '1B': 4, '2B': 0, '3B': 0, 'HR': 0, 'RBI': 0, 'BB': 1, 'K': 10, 'AVG': 0.4, 'OBP': 0.208, 'SLG': 0.4, 'OPS': 0.608}

Iván Herrera:
{'Matchup': 'Iván Herrera vs Miles Mikolas', 'PA': 0, 'AB': 0, 'H': 0, '1B': 0, '2B': 0, '3B': 0, 'HR': 0, 'RBI': 0, 'BB': 0, 'K': 0, 'AVG': 0, 'OBP': 0, 'SLG': 0, 'OPS': 0}

Christian Walker:
{'Matchup': 'Christian Walker vs Michael Lorenzen', 'PA': 10, 'AB': 7, 'H': 2, '1B': 1, '2B': 1, '3B': 0, 'HR': 0, 'RBI': 0, 'BB': 2, 'K': 1, 'AVG': 0.286, 'OBP': 0.4, 'SLG': 0.429, 'OPS': 0.829}

Matt Mclain:
{'Matchup': 'Matt Mclain vs Eury Pérez', 'PA': 2, 'AB': 1, 'H': 1, '1B': 0, '2B': 0, '3B': 0, 'HR': 1, 'RBI': 0, 'BB': 0, 'K': 1, 'AVG': 1.0, 'OBP': 